In [37]:
# Chunk by a set number of characters
def chunk_by_char(text, chunk_size=150, chunk_overlap=20):
    chunks = []
    start_idx = 0

    while start_idx < len(text):
        # Calculate end of this chunk, capping at end of text
        end_idx = min(start_idx + chunk_size, len(text))

        # Slice out the chunk and store it
        chunk_text = text[start_idx:end_idx]
        chunks.append(chunk_text)

        # Move start forward by chunk_size, but step back by overlap so the
        # next chunk shares chunk_overlap characters with the current one.
        # If we've reached the end of the text, jump to len(text) to exit the loop.
        start_idx = (
            end_idx - chunk_overlap if end_idx < len(text) else len(text)
        )

    return chunks

In [38]:
# Chunk by sentence
import re

def chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1):
    # Split on whitespace that follows a sentence-ending punctuation mark (. ! ?)
    # The lookbehind (?<=[.!?]) ensures we only split after those characters,
    # not on every whitespace, so sentences stay intact.
    sentences = re.split(r"(?<=[.!?])\s+", text)

    chunks = []
    start_idx = 0

    while start_idx < len(sentences):
        # Take up to max_sentences_per_chunk sentences, stopping at end of list
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))

        # Join the selected sentences back into a single string for this chunk
        current_chunk = sentences[start_idx:end_idx]
        chunks.append(" ".join(current_chunk))

        # Advance by (max_sentences - overlap) so the next chunk re-uses the
        # last overlap_sentences sentences, preserving context at boundaries
        start_idx += max_sentences_per_chunk - overlap_sentences

        # Guard against negative index if overlap > max_sentences
        if start_idx < 0:
            start_idx = 0

    return chunks

In [ ]:
def chunk_by_section(document_text: str) -> list[dict]:
    """
    Splits markdown text into sections using heading markers (## / ### / ####).

    Splits on levels 2-4 only:
      - Level 1 (#) is the document title — treated as preamble, not a boundary.
      - Levels 2-4 map to real sections and subsections.
      - Level 5+ (#####) are pull quotes or callouts in some PDFs — ignored.

    pymupdf4llm assigns heading levels based on font size, so this reliably
    identifies document sections without hardcoding document-specific patterns.
    """
    pattern = re.compile(r"\n(?=#{2,4} )")
    raw_chunks = pattern.split(document_text)
    sections = []
    for chunk in raw_chunks:
        chunk = chunk.strip()
        if not chunk:
            continue
        lines = chunk.splitlines()
        heading_line = lines[0].lstrip("#").strip()
        body = "\n".join(lines[1:]).strip()
        sections.append({"title": heading_line, "content": body})
    return sections

In [ ]:
import re
import json
import pymupdf4llm
from pathlib import Path

DATA_DIR = Path("data")
CHUNKS_OUTPUT = DATA_DIR / "chunks.jsonl"
CHROMA_PATH = DATA_DIR / "chroma"
COLLECTION_NAME = "regulatory_docs"

MIN_CHUNK_CHARS = 100
EMBED_MODEL = "voyage-3-large"
BATCH_SIZE = 128  # Voyage AI max texts per request

In [41]:
def clean_text(text: str) -> str:
    text = re.sub(r"<sup>.*?</sup>", "", text)        # strip footnote superscripts
    text = re.sub(r"\*{1,2}(.+?)\*{1,2}", r"\1", text)  # strip bold/italic markdown
    text = re.sub(r"\n{3,}", "\n\n", text)               # collapse excessive blank lines
    return text.strip()


def make_chunk_id(filename_stem: str, title: str, index: int) -> str:
    slug = re.sub(r"[^a-z0-9]+", "_", title.lower()).strip("_")
    return f"{filename_stem}__{slug}__{index}"


def process_pdf(pdf_path: Path) -> list[dict]:
    markdown = pymupdf4llm.to_markdown(str(pdf_path))
    raw_sections = chunk_by_section(markdown)
    chunks = []
    for i, section in enumerate(raw_sections):
        text = clean_text(section["content"])
        if len(text) < MIN_CHUNK_CHARS:
            continue
        chunks.append({
            "id": make_chunk_id(pdf_path.stem, section["title"], i),
            "text": text,
            "metadata": {
                "source": pdf_path.name,
                "section": clean_text(section["title"]),
            },
        })
    return chunks

In [42]:
all_chunks = []

for pdf_path in sorted(DATA_DIR.glob("*.pdf")):
    chunks = process_pdf(pdf_path)
    all_chunks.extend(chunks)
    print(f"{pdf_path.name}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

with CHUNKS_OUTPUT.open("w", encoding="utf-8") as f:
    for chunk in all_chunks:
        f.write(json.dumps(chunk) + "\n")

print(f"Saved to {CHUNKS_OUTPUT}")

bcbs-239-rdar.pdf: 19 chunks
boe-financial-stability-in-focus-artificial-intelligence.pdf: 7 chunks
boe-fs2-23-artificial-intelligence-and-machine-learning.pdf: 26 chunks
pra-ss1-23-model-risk.pdf.pdf: 46 chunks

Total chunks: 98
Saved to data\chunks.jsonl


In [43]:
# Preview a sample chunk
import pprint
pprint.pprint(all_chunks[10])

{'id': 'bcbs-239-rdar__iii_risk_reporting_practices__10',
 'metadata': {'section': 'III. Risk reporting practices',
              'source': 'bcbs-239-rdar.pdf'},
 'text': '51. Accurate, complete and timely data is a foundation for effective '
         'risk management. However, data alone does not guarantee that the '
         'board and senior management will receive appropriate information to '
         'make effective decisions about risk. To manage risk effectively, the '
         'right information needs to be presented to the right people at the '
         'right time. Risk reports based on risk data should be accurate, '
         'clear and complete. They should contain the correct content and be '
         'presented to the appropriate decision-makers in a time that allows '
         'for an appropriate response. To effectively achieve their '
         'objectives, risk reports should comply with the following '
         'principles. Compliance with these principles should not 

In [ ]:
import voyageai
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

vo = voyageai.Client()


def embed_chunks(chunks: list[dict], model: str = EMBED_MODEL) -> list[dict]:
    """
    Embeds a list of chunk dicts using Voyage AI and returns them with an
    added 'embedding' key containing the vector (list of floats).

    Sends texts in batches of BATCH_SIZE (128, the Voyage AI maximum per request).
    input_type='document' produces vectors optimised for retrieval storage
    rather than query matching (use input_type='query' at search time).
    """
    texts = [c["text"] for c in chunks]
    embeddings = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i : i + BATCH_SIZE]
        result = vo.embed(batch, model=model, input_type="document")
        embeddings.extend(result.embeddings)
        print(f"  Embedded {min(i + BATCH_SIZE, len(texts))}/{len(texts)}")
    for chunk, vector in zip(chunks, embeddings):
        chunk["embedding"] = vector
    return chunks


all_chunks = embed_chunks(all_chunks)
print(f"\nDone — {len(all_chunks)} chunks embedded")

In [ ]:
import chromadb

# Persistent client stores the database on disk so embeddings survive between sessions.
# The directory is created automatically if it doesn't exist.
chroma_client = chromadb.PersistentClient(path=str(CHROMA_PATH))

# get_or_create_collection reuses an existing collection on re-runs,
# preventing duplicate inserts if this cell is run more than once.
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},  # cosine similarity matches Voyage AI's embedding space
)

collection.add(
    ids=[c["id"] for c in all_chunks],
    embeddings=[c["embedding"] for c in all_chunks],
    documents=[c["text"] for c in all_chunks],
    metadatas=[c["metadata"] for c in all_chunks],
)

print(f"Ingested {collection.count()} documents into '{COLLECTION_NAME}' at {CHROMA_PATH}")